# DMSF Training — Kaggle

**Chuẩn bị trước khi chạy:**

1. Zip folder `baseline_DMSF/` → upload lên [Kaggle Datasets](https://www.kaggle.com/datasets) → slug: `baseline-dmsf`
2. Tìm dataset VisDrone YOLO trên Kaggle (search `visdrone yolo`) → Add vào notebook
   - Nếu không có: điền `GDRIVE_FOLDER_ID` trong Cell 3 (Drive → share folder visdrone/ → copy folder ID)
3. Settings → Accelerator: **GPU T4 x1**, Internet: **On**
4. Run All (Cell 1 → 4)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Load code DMSF
# Upload baseline_DMSF.zip lên Kaggle Datasets (slug: baseline-dmsf)
# Kaggle tự giải nén → tìm folder baseline_DMSF trong /kaggle/input/
# ═══════════════════════════════════════════════════════════════
import os, sys, shutil

CODE_DIR = '/kaggle/working/baseline_DMSF'

if not os.path.isdir(CODE_DIR):
    # Kaggle đã giải nén sẵn → tìm folder baseline_DMSF
    SRC = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == 'baseline_DMSF':
            SRC = root
            break
        # Fallback: vẫn còn file zip chưa giải nén
        for f in files:
            if 'baseline' in f.lower() and f.endswith('.zip'):
                print(f'Found zip: {f}, extracting...')
                !unzip -q "{os.path.join(root, f)}" -d /kaggle/working/
                SRC = None
                break
        if SRC or os.path.isdir(CODE_DIR):
            break

    if SRC:
        shutil.copytree(SRC, CODE_DIR)
        print(f'✓ Copied from {SRC}')

    assert os.path.isdir(CODE_DIR), (
        '❌ Không tìm thấy baseline_DMSF\n'
        '   → Add dataset baseline-dmsf vào notebook (+ → Add Data)'
    )
    print('✓ Code extracted')
else:
    print('✓ Code đã có')

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print(f'Working dir: {os.getcwd()}')
!ls

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Chuẩn bị VisDrone data
# Hỗ trợ 2 loại dataset:
#   A) Raw VisDrone (có VisDrone2019-DET-train/) → tự convert sang YOLO
#   B) Pre-converted YOLO (có images/train/)     → copy thẳng
# ═══════════════════════════════════════════════════════════════
import os, shutil

DATA_DIR  = 'data/visdrone'   # relative to CODE_DIR (/kaggle/working/baseline_DMSF)
DONE_FLAG = DATA_DIR + '/.done'

def _count(split):
    p = f'{DATA_DIR}/images/{split}'
    return len(os.listdir(p)) if os.path.isdir(p) else 0

if os.path.exists(DONE_FLAG) and _count('train') > 0:
    print(f'✓ VisDrone ready: {_count("train")} train | {_count("val")} val')

else:
    KAGGLE_SRC = None
    RAW_SRC    = None   # thư mục chứa VisDrone2019-DET-train/

    for root, dirs, files in os.walk('/kaggle/input'):
        # Option B: đã có YOLO format (images/train/)
        if 'images' in dirs and os.path.isdir(os.path.join(root, 'images', 'train')):
            KAGGLE_SRC = root
            break
        # Option A: raw VisDrone format
        if any('VisDrone2019-DET-train' in d for d in dirs):
            RAW_SRC = root
            break

    if KAGGLE_SRC:
        print(f'Found YOLO dataset: {KAGGLE_SRC}')
        os.makedirs(DATA_DIR, exist_ok=True)
        !cp -r "{KAGGLE_SRC}/images" "{DATA_DIR}/"
        !cp -r "{KAGGLE_SRC}/labels" "{DATA_DIR}/"

    elif RAW_SRC:
        print(f'Found raw VisDrone: {RAW_SRC}')
        print('Converting annotations to YOLO format...')
        from utils.visdrone import prepare_visdrone
        prepare_visdrone(src_root=RAW_SRC, dst_root=DATA_DIR, splits=('train', 'val'))

    else:
        raise AssertionError(
            '❌ Không tìm thấy VisDrone dataset trong /kaggle/input/\n'
            '   → Add dataset VisDrone vào notebook (+ Add Input)'
        )

    assert _count('train') > 0, '❌ Convert thất bại — kiểm tra lại dataset'
    open(DONE_FLAG, 'w').close()
    print(f'✓ Done: {_count("train")} train | {_count("val")} val')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: Training (auto-resume từ checkpoint)
# Muốn train lại từ đầu: xoá last.pt và best.pt trong
#   /kaggle/working/checkpoints/ rồi commit + chạy lại
# ═══════════════════════════════════════════════════════════════
import os, sys, shutil, time, random, urllib.request
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from pathlib import Path

from models.dmsf import DMSF
from utils.visdrone import VisDroneDataset
from utils.loss import ComputeLoss
from utils.metrics import evaluate

EPOCHS    = 600
BATCH     = 32
IMGSZ     = 640
NC        = 10
LR        = 0.01
WORKERS   = 2
VAL_FREQ  = 10
SAVE_FREQ = 10
DEVICE    = torch.device('cuda:0')
CKPT_DIR  = '/kaggle/working/checkpoints'
LOCAL_DIR = Path('runs/train')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

model = DMSF(nc=NC).to(DEVICE)
start_epoch  = 0
best_map50   = 0.0
best_map5095 = 0.0
ckpt = None

ckpt_last  = Path(CKPT_DIR) / 'last.pt'
local_last = LOCAL_DIR / 'last.pt'

if ckpt_last.exists():
    shutil.copy(ckpt_last, local_last)
    ckpt = torch.load(local_last, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'], strict=False)
    start_epoch  = ckpt.get('epoch', 0)
    best_map50   = ckpt.get('map50', 0.0)
    best_map5095 = ckpt.get('map50_95', 0.0)
    print(f'↓ Resumed checkpoint')
    print(f'▶ Resuming từ epoch {start_epoch}  '
          f'(best mAP@50={best_map50:.4f}  mAP@50:95={best_map5095:.4f})')
else:
    pretrained = '/kaggle/working/yolov5s.pt'
    MIN_SIZE   = 10 * 1024 * 1024  # 10 MB
    if not os.path.exists(pretrained) or os.path.getsize(pretrained) < MIN_SIZE:
        print('Downloading pretrained YOLOv5s...')
        url = 'https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt'
        urllib.request.urlretrieve(url, pretrained)
        print(f'  Downloaded: {os.path.getsize(pretrained)/1e6:.1f} MB')
    model.load_yolov5s_weights(pretrained)
    print('▶ Train từ đầu với pretrained YOLOv5s backbone')

train_ds = VisDroneDataset('data/visdrone/images/train', IMGSZ, augment=True)
val_ds   = VisDroneDataset('data/visdrone/images/val',   IMGSZ, augment=False)

# ── Bỏ comment 2 dòng dưới để test nhanh (~25s/epoch thay vì 300s) ──
# train_ds = Subset(train_ds, range(500))
# val_ds   = Subset(val_ds,   range(100))

train_loader = DataLoader(train_ds, BATCH, shuffle=True,  num_workers=WORKERS,
                          pin_memory=True, persistent_workers=True,
                          collate_fn=VisDroneDataset.collate_fn, drop_last=True)
val_loader   = DataLoader(val_ds,   BATCH//2, shuffle=False, num_workers=2,
                          pin_memory=True, persistent_workers=True,
                          collate_fn=VisDroneDataset.collate_fn)

pg0, pg1, pg2 = [], [], []
for n, p in model.named_parameters():
    if not p.requires_grad: continue
    if '.bias'   in n:                      pg2.append(p)
    elif '.weight' in n and '.bn' not in n: pg1.append(p)
    else:                                   pg0.append(p)

optimizer = optim.SGD(pg0, lr=LR, momentum=0.937, nesterov=True)
optimizer.add_param_group({'params': pg1, 'weight_decay': 5e-4})
optimizer.add_param_group({'params': pg2, 'weight_decay': 0.0})

if ckpt is not None and 'optimizer' in ckpt:
    optimizer.load_state_dict(ckpt['optimizer'])

scheduler = optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda x: (1 - x / EPOCHS) * 0.9 + 0.1,
    last_epoch=start_epoch - 1
)
compute_loss = ComputeLoss(model, nc=NC)

print(f'\nBắt đầu train: epoch {start_epoch+1} → {EPOCHS}')
print(f'Dataset: {len(train_ds)} train | {len(val_ds)} val  |  Batch={BATCH}')
print('─' * 65)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for imgs, targets, _ in train_loader:
        imgs    = imgs.to(DEVICE).float()
        targets = targets.to(DEVICE)
        sp = random.choice(model.split_points)
        optimizer.zero_grad()
        loss, _ = compute_loss(model(imgs, split_point=sp), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    elapsed  = time.time() - t0
    print(f'Epoch {epoch+1:4d}/{EPOCHS}  loss={avg_loss:.4f}  '
          f'lr={optimizer.param_groups[0]["lr"]:.5f}  {elapsed:.0f}s', flush=True)

    if (epoch + 1) % VAL_FREQ == 0:
        m = evaluate(model, val_loader, DEVICE, split_point=10, img_size=IMGSZ)
        print(f'  ↳ [Val] mAP@50={m["map50"]:.4f}  mAP@50:95={m["map50_95"]:.4f}')
        if m['map50'] > best_map50:
            best_map50   = m['map50']
            best_map5095 = m['map50_95']
            torch.save({'epoch': epoch+1, 'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'map50': best_map50, 'map50_95': best_map5095},
                       LOCAL_DIR / 'best.pt')
            shutil.copy(LOCAL_DIR / 'best.pt', f'{CKPT_DIR}/best.pt')
            print(f'  ↳ ✓ best.pt saved  '
                  f'(mAP@50={best_map50:.4f}  mAP@50:95={best_map5095:.4f})')
        model.train()

    if (epoch + 1) % SAVE_FREQ == 0:
        torch.save({'epoch': epoch+1, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'map50': best_map50, 'map50_95': best_map5095},
                   LOCAL_DIR / 'last.pt')
        shutil.copy(LOCAL_DIR / 'last.pt', f'{CKPT_DIR}/last.pt')
        print(f'  ↳ ✓ Checkpoint saved (epoch {epoch+1})')

print(f'\n✓ Training hoàn tất. Best mAP@50={best_map50:.4f}  mAP@50:95={best_map5095:.4f}')

## Download model

Sau khi training xong, vào **Output** (góc phải) → download `checkpoints/best.pt` và `checkpoints/last.pt`.

Hoặc commit notebook để Kaggle lưu output vĩnh viễn.